# EDA Dataset SGCC LARGE (window=365, stride=14)

Dataset SGCC penuh: **42,372 consumer × 1,034 hari** konsumsi listrik harian (kWh).  
Label per consumer: `FLAG=0` = normal, `FLAG=1` = pencurian listrik.

Pipeline ini menggunakan **window_size=365** (1 tahun penuh) dan **stride=14** (geser 2 minggu),
menghasilkan **~48 windows per consumer**. Stratified sampling ~2,000 consumer menjaga rasio fraud/normal.

In [ ]:
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

CARLA_ROOT = Path('..').resolve()
if str(CARLA_ROOT) not in sys.path:
    sys.path.insert(0, str(CARLA_ROOT))

RAW_ZIP        = CARLA_ROOT / 'datasets' / 'SGCC' / 'raw' / 'data set.csv.zip'
PROCESSED_LARGE= CARLA_ROOT / 'datasets' / 'SGCC' / 'processed_large'
FIG_DIR        = CARLA_ROOT / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 365
STRIDE      = 14

print(f'CARLA root     : {CARLA_ROOT}')
print(f'Raw ZIP        : {RAW_ZIP}  exists={RAW_ZIP.exists()}')
print(f'Processed large: {PROCESSED_LARGE}  exists={PROCESSED_LARGE.exists()}')
print(f'window_size={WINDOW_SIZE}  stride={STRIDE}')


## 1. Load raw data

In [ ]:
print("Membaca data set.csv.zip (~163 MB unzipped) ...")
with zipfile.ZipFile(RAW_ZIP) as zf:
    csv_names = [n for n in zf.namelist() if n.endswith('.csv')]
    with zf.open(csv_names[0]) as f:
        df = pd.read_csv(f)

META_COLS = {'CONS_NO', 'FLAG'}
date_cols = [c for c in df.columns if c not in META_COLS]
T = len(date_cols)
N = len(df)

print(f'\nDimensi dataset:')
print(f'  Consumer (baris): {N:,}')
print(f'  Hari (kolom)    : {T}  ({date_cols[0]} .. {date_cols[-1]})')
print(f'  Total nilai     : {N*T:,}')

n_fraud  = int((df['FLAG'] == 1).sum())
n_normal = int((df['FLAG'] == 0).sum())
print(f'\nLabel:')
print(f'  Normal (0): {n_normal:,}  ({100*n_normal/N:.2f}%)')
print(f'  Fraud  (1): {n_fraud:,}   ({100*n_fraud/N:.2f}%)')


## 2. Missing Values

In [ ]:
nan_total = df[date_cols].isna().sum().sum()
nan_pct   = nan_total / (N * T) * 100
nan_per_row = df[date_cols].isna().sum(axis=1)

print(f'Total NaN  : {nan_total:,}  ({nan_pct:.2f}% dari semua nilai)')
print(f'Consumer dengan ≥1 NaN: {(nan_per_row > 0).sum():,}  ({(nan_per_row>0).mean()*100:.1f}%)')
print(f'Consumer dengan >100 NaN: {(nan_per_row > 100).sum():,}')
print(f'Consumer dengan >500 NaN: {(nan_per_row > 500).sum():,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribusi NaN per consumer
axes[0].hist(nan_per_row, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Jumlah NaN per Consumer')
axes[0].set_ylabel('Jumlah Consumer')
axes[0].set_title(f'Distribusi NaN per Consumer\n(total NaN: {nan_total:,}, {nan_pct:.1f}%)')
axes[0].axvline(nan_per_row.median(), color='red', linestyle='--',
                label=f'Median={nan_per_row.median():.0f}')
axes[0].legend()

# NaN per kolom (hari) — rata-rata 1000 consumer
sample_idx = np.random.RandomState(42).choice(N, min(1000, N), replace=False)
nan_per_col = df.iloc[sample_idx][date_cols].isna().mean()
axes[1].plot(range(T), nan_per_col.values, color='darkorange', linewidth=0.7)
axes[1].set_xlabel('Indeks Hari (0 = 1/1/2014)')
axes[1].set_ylabel('Fraksi NaN')
axes[1].set_title(f'Fraksi NaN per Hari (sample 1000 consumer)')
axes[1].set_xlim(0, T)

plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_sgcc_large_nan.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan.')


## 3. Distribusi Label

In [ ]:
counts = df['FLAG'].value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['Normal (0)', 'Fraud (1)'], [n_normal, n_fraud],
            color=['#2196F3', '#F44336'], edgecolor='white', width=0.5)
axes[0].set_ylabel('Jumlah Consumer')
axes[0].set_title(f'Distribusi Label SGCC LARGE\n(Total: {N:,} consumer)')
for i, v in enumerate([n_normal, n_fraud]):
    axes[0].text(i, v + 200, f'{v:,}\n({v/N*100:.1f}%)', ha='center', fontsize=10)

axes[1].pie([n_normal, n_fraud], labels=['Normal', 'Fraud'],
            colors=['#2196F3', '#F44336'], autopct='%1.1f%%',
            startangle=140, pctdistance=0.8)
axes[1].set_title('Proporsi Label')

plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_sgcc_large_label_dist.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Distribusi kWh (raw)

In [ ]:
# Sample 500 consumer dari setiap kelas untuk efisiensi
rng = np.random.RandomState(42)
normal_idx = df.index[df['FLAG'] == 0].tolist()
fraud_idx  = df.index[df['FLAG'] == 1].tolist()
rng.shuffle(normal_idx); rng.shuffle(fraud_idx)

n_sample = 300
normal_vals = df.iloc[normal_idx[:n_sample]][date_cols].values.astype(float).flatten()
fraud_vals  = df.iloc[fraud_idx[:min(n_sample, len(fraud_idx))]][date_cols].values.astype(float).flatten()
normal_vals = normal_vals[~np.isnan(normal_vals) & (normal_vals < 200)]  # hapus outlier ekstrem
fraud_vals  = fraud_vals[~np.isnan(fraud_vals)   & (fraud_vals  < 200)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bins = np.linspace(0, 50, 100)
axes[0].hist(normal_vals, bins=bins, alpha=0.7, color='#2196F3', label=f'Normal (n={len(normal_vals):,})', density=True)
axes[0].hist(fraud_vals,  bins=bins, alpha=0.7, color='#F44336', label=f'Fraud (n={len(fraud_vals):,})',  density=True)
axes[0].set_xlabel('Konsumsi harian (kWh)')
axes[0].set_ylabel('Densitas')
axes[0].set_title('Distribusi kWh Normal vs Fraud\n(0–50 kWh, raw, tanpa normalisasi)')
axes[0].legend()

# Log scale
axes[1].hist(normal_vals + 0.01, bins=np.logspace(-2, 2, 80), alpha=0.7,
             color='#2196F3', label='Normal', density=True)
axes[1].hist(fraud_vals + 0.01,  bins=np.logspace(-2, 2, 80), alpha=0.7,
             color='#F44336', label='Fraud',  density=True)
axes[1].set_xscale('log')
axes[1].set_xlabel('Konsumsi harian (kWh, log scale)')
axes[1].set_ylabel('Densitas')
axes[1].set_title('Distribusi kWh (log scale)')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_sgcc_large_kwh_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Normal: median={np.median(normal_vals):.2f}  mean={np.mean(normal_vals):.2f}  std={np.std(normal_vals):.2f} kWh")
print(f"Fraud : median={np.median(fraud_vals):.2f}  mean={np.mean(fraud_vals):.2f}  std={np.std(fraud_vals):.2f} kWh")


## 5. Profil Time Series: Normal vs Fraud (sampel)

In [ ]:
def fill_nan_series(s):
    s = pd.Series(s, dtype=float)
    s = s.interpolate(method='linear', limit_direction='both')
    return s.ffill().bfill().fillna(0.0).values

fig, axes = plt.subplots(4, 2, figsize=(16, 14))
pairs = [(normal_idx, 'Normal', '#2196F3'), (fraud_idx, 'Fraud', '#F44336')]

date_range = pd.date_range('2014-01-01', periods=T, freq='D')

for col_idx, (idx_list, label, color) in enumerate(pairs):
    for row_idx in range(4):
        ax = axes[row_idx][col_idx]
        cons_idx = idx_list[row_idx]
        cons_id  = str(df.loc[cons_idx, 'CONS_NO'])[:12]
        ts_raw   = df.loc[cons_idx, date_cols].values.astype(float)
        ts_filled= fill_nan_series(ts_raw)

        ax.fill_between(date_range, ts_filled, alpha=0.3, color=color)
        ax.plot(date_range, ts_filled, color=color, linewidth=0.8)

        # Tandai jendela 365 hari pertama
        ax.axvline(date_range[0],   color='gray', linestyle=':', linewidth=1)
        ax.axvline(date_range[364], color='gray', linestyle=':', linewidth=1, label='window 0 (365 hari)')

        ax.set_title(f'{label} — {cons_id}...', fontsize=9)
        ax.set_xlabel('')
        ax.set_ylabel('kWh/hari')
        if row_idx == 0:
            ax.legend(fontsize=7)

plt.suptitle('Profil kWh Harian: Normal vs Pencurian (4 sampel masing-masing)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_sgcc_large_profiles.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Window Analysis (window=365, stride=14)

In [ ]:
# Analisis matematis windowing
T_val = T  # 1034
WS = WINDOW_SIZE  # 365
ST = STRIDE       # 14

n_windows = (T_val - WS) // ST + 1
print(f'T          = {T_val} hari')
print(f'window_size= {WS}')
print(f'stride     = {ST}')
print(f'n_windows per consumer = ({T_val} - {WS}) // {ST} + 1 = {n_windows}')

print(f'\nEstimasi dataset size untuk 2000 consumer (stratified):')
n_train_consumers = round(2000 * 0.8 * (1 - 0.0853))  # ~80% dari 91.47% normal
n_windows_train   = n_train_consumers * n_windows
print(f'  Train consumers (normal, ~80%)  : ~{n_train_consumers}')
print(f'  Train windows                   : ~{n_windows_train:,}')
print(f'  Con_data dgn aug (×2)           : ~{n_windows_train*2:,}')
print(f'  Batches/epoch (batch_size=256)  : ~{n_windows_train*2//256}')

# Visualisasi: windows pada satu consumer
fig, ax = plt.subplots(figsize=(14, 5))
example_ts = fill_nan_series(df.loc[normal_idx[0], date_cols].values.astype(float))
ax.plot(date_range, example_ts, color='#2196F3', linewidth=0.8, alpha=0.7, label='kWh harian')

# Gambar beberapa jendela pertama dan terakhir
n_show = 5
colors_w = plt.cm.Reds(np.linspace(0.3, 0.9, n_show))
for i in range(n_show):
    start = i * ST
    end   = start + WS
    ax.axvspan(date_range[start], date_range[min(end, T_val-1)],
               alpha=0.15, color=colors_w[i], label=f'Window {i}' if i < 3 else None)

last_start = (n_windows - 1) * ST
ax.axvspan(date_range[last_start], date_range[min(last_start+WS, T_val-1)],
           alpha=0.25, color='green', label=f'Window {n_windows-1} (terakhir)')

ax.set_xlabel('Tanggal')
ax.set_ylabel('kWh/hari')
ax.set_title(f'Ilustrasi Windowing (W={WS}, s={ST}) — Consumer Normal\n'
             f'{n_windows} windows total per consumer, masing-masing 1 tahun penuh')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_sgcc_large_windows.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Stratified Sampling Strategy

In [ ]:
# Simulasi stratified sampling
import numpy as np

N_total   = N
n_fraud_t = n_fraud
n_normal_t= n_normal
fraud_ratio = n_fraud_t / N_total

print(f'Dataset penuh:')
print(f'  Total   : {N_total:,}')
print(f'  Normal  : {n_normal_t:,}  ({100*(1-fraud_ratio):.2f}%)')
print(f'  Fraud   : {n_fraud_t:,}   ({100*fraud_ratio:.2f}%)')
print()

targets = [500, 1000, 1500, 2000, 3000, 5000]
print(f'  {"Target":>8}  {"Normal":>8}  {"Fraud":>8}  {"Ratio":>8}  {"Train W":>10}  {"CData":>10}  {"Est. min (cls)":>14}')
print(f'  {"-"*80}')
for t in targets:
    nf = round(t * fraud_ratio)
    nn = t - nf
    n_train = round(nn * 0.8)
    n_win   = n_train * n_windows
    cdata   = n_win * 2
    batches = cdata // 256
    # est: 0.144s/batch (dari benchmark small dataset), 15 epoch
    est_min = batches * 0.144 * 15 / 60
    print(f'  {t:>8,}  {nn:>8,}  {nf:>8,}  {nf/t*100:>7.2f}%  {n_win:>10,}  {cdata:>10,}  {est_min:>12.1f} min')

print()
print('→ Rekomendasi: 2000 consumer (~47 menit total) — aman di bawah 1 jam')
print('→ Untuk lebih cepat: 1000-1500 consumer (~20-30 menit)')


## 8. Verifikasi Data Setelah Preprocess (jika processed_large/ sudah ada)

In [ ]:
import os

train_list_path = PROCESSED_LARGE / 'train_list.txt'
if not train_list_path.exists():
    print('processed_large/ belum ada. Jalankan: python preprocess_sgcc_large.py')
    print('(atau python run_sgcc_large.py)')
else:
    with open(PROCESSED_LARGE / 'train_list.txt') as f:
        train_files = [l.strip() for l in f if l.strip()]
    with open(PROCESSED_LARGE / 'val_list.txt') as f:
        val_files = [l.strip() for l in f if l.strip()]
    with open(PROCESSED_LARGE / 'test_list.txt') as f:
        test_files = [l.strip() for l in f if l.strip()]

    import csv as csv_mod
    labels = {}
    with open(PROCESSED_LARGE / 'file_labels.csv', newline='') as f:
        for row in csv_mod.DictReader(f):
            labels[row['fname']] = int(row['label'])

    all_files = train_files + val_files + test_files
    n_total   = len(all_files)
    n_fraud_p = sum(1 for f in all_files if labels.get(f, 0) == 1)
    n_normal_p= n_total - n_fraud_p

    print(f'processed_large/ tersedia!')
    print(f'  total consumer: {n_total}')
    print(f'  train_list.txt: {len(train_files)} (normal saja)')
    print(f'  val_list.txt  : {len(val_files)}')
    print(f'  test_list.txt : {len(test_files)}')
    print(f'  val+test fraud: {n_fraud_p}  ({100*n_fraud_p/n_total:.2f}%)')

    # Cek satu .npy
    sample_npy = PROCESSED_LARGE / 'data' / all_files[0]
    arr = np.load(sample_npy)
    print(f'  Shape .npy sample ({all_files[0][:20]}...): {arr.shape}')
    print(f'  dtype: {arr.dtype}  min={arr.min():.3f}  max={arr.max():.3f} kWh')
    assert arr.shape == (T, 1), f'Expected ({T},1), got {arr.shape}'
    print(f'  ✓ Shape konsisten dengan T={T} hari')

    # Estimasi windows
    n_win_per = (arr.shape[0] - WINDOW_SIZE) // STRIDE + 1
    print(f'  Windows per consumer (W={WINDOW_SIZE}, s={STRIDE}): {n_win_per}')
    print(f'  Total train windows: {len(train_files) * n_win_per:,}')


## 9. Ringkasan Pipeline

In [ ]:
print('=' * 65)
print('RINGKASAN PIPELINE SGCC LARGE → CARLA')
print('=' * 65)
print()
print(f'Dataset    : SGCC besar (data set.csv.zip, 163 MB)')
print(f'Consumers  : {N:,} total  ({n_normal:,} normal + {n_fraud:,} fraud)')
print(f'Hari/cons  : {T} hari (1 Jan 2014 — 31 Okt 2016)')
print()
print(f'Windowing  :')
print(f'  window_size = {WINDOW_SIZE} hari (1 tahun kalender)')
print(f'  stride      = {STRIDE} hari (2 minggu)')
print(f'  windows/con = {(T - WINDOW_SIZE) // STRIDE + 1}')
print()
print(f'Sampling   :')
print(f'  Target     = 2000 consumer (stratified)')
print(f'  Fraud ratio ≈ 8.53% (dijaga sama dengan aslinya)')
print()
print('Cara menjalankan:')
print('  1. python preprocess_sgcc_large.py        # ~5 menit')
print('  2. python run_sgcc_large.py               # ~47 menit total')
print('     -- atau --')
print('  python run_sgcc_large.py                  # preprocess + training otomatis')
print()
print('Evaluasi saja (setelah training):')
print('  python evaluate.py --dataset SGCC_LARGE')
print('=' * 65)
